In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import itertools

In [ ]:
th = 24.68
th *= (np.pi)/180
theta_0 = .6*th

In [ ]:
dU   = 1.0254384902642049#np.sqrt(np.log(2))/(np.pi*theta_0)
Umax = 602
NN   = int(2*np.ceil(Umax/dU))
Cxy  = dU
ff   = 0.00025

Nbin = 25
Nmax = NN//3 #392
Nr   = Nbin

beam_norm_fact = 3./(np.pi*theta_0*theta_0)



In [ ]:
Nmax*dU

In [ ]:
kb = 1.38*1e-23
nu_sq=nu = (154.255*1e6)**2
c_sq = (3e8)**2
Q_cube = (2*kb*nu_sq/c_sq)**3

In [ ]:
Q_cube

In [ ]:
currentNumber = 1
uLow = [0] * Nbin
uHigh = [0] * Nbin

intervals = [(0, 50, 5), (50, 100, 10),(100, 200, 20), (200, 305, 35), (305, 393, 44)]

# Index for uLow array
index = 0

for start, end, step in intervals:
    for i in range(start, end, step):
        if index < Nbin:
            uLow[index] = i
            if index > 0:
                uHigh[index - 1] = uLow[index]
            index += 1

# Handle the last uHigh value if it hasn't been set
if index > 0 and index < Nbin:
    uHigh[index - 1] = Nmax

uLow[0] = 1   
uHigh[Nbin - 1] = 1.0 * Nmax

uavg=(2/3)*(np.array(uHigh)**3-np.array(uLow)**3)/(np.array(uHigh)**2-np.array(uLow)**2)
# Print the arrays for verification
print("uLow:", uLow)
print("uHigh:", uHigh)
print("Diff:", np.array(uHigh)-np.array(uLow))

In [ ]:

uval = dU*uavg
def stack_data(rlzn):
    res = np.loadtxt("%s/bsout_u%d_%.3f_GV_data_%d"%(loc,0,uval[0],rlzn))
    for u1 in range(1,Nbin):
        #bs_df = pd.read_csv(r'%s/bsout_u%d_%.3f_GV_data_%d'%(loc,u1,uval[u1],rlzn),sep=" ",names=["u1","t","mu","bs","bana","error","tri"])
        bs_data = np.loadtxt("%s/bsout_u%d_%.3f_GV_data_%d"%(loc,u1,uval[u1],rlzn))
        res = np.vstack((res, bs_data))
    return res.T

def get_bin_number(values, num_bins):
    values = np.asarray(values)
    bin_edges = np.linspace(0.5, 1.0, num_bins + 1)
    bin_numbers = np.digitize(values, bin_edges, right=False) - 1
    bin_numbers = np.where(bin_numbers == num_bins, num_bins - 1, bin_numbers)
    return bin_numbers

In [ ]:
350*2*np.pi

In [ ]:
def binned_data(rlzn,num_bins):
#     bs_mut = [[[] for _ in range(num_bins)] for _ in range(num_bins)]
    bs_mut = np.empty((num_bins,num_bins), dtype=object)
    bs_data = stack_data(rlzn)
    t_bins  = get_bin_number(bs_data[1],num_bins)
    mu_bins = get_bin_number(bs_data[2],num_bins)
    for i, j in itertools.product(range(num_bins), range(num_bins)):
        mu_t_bin= np.where((mu_bins == i) & (t_bins == j))   # (t_bins==4)*(mu_bins==0)
        bs_mut[i][j] = bs_data[3][mu_t_bin]
    return bs_mut

In [ ]:
def binned_ana_data(num_bins):
    u1_mut = [[[] for _ in range(num_bins)] for _ in range(num_bins)]
    tri_mut = [[[] for _ in range(num_bins)] for _ in range(num_bins)]
    ana_mut = [[[] for _ in range(num_bins)] for _ in range(num_bins)]
    
    bs_data = stack_data(1)
    t_bins  = get_bin_number(bs_data[1],num_bins)
    mu_bins = get_bin_number(bs_data[2],num_bins)
    for i, j in itertools.product(range(num_bins), range(num_bins)):
        mu_t_bin= np.where((mu_bins == i) & (t_bins == j))   # (t_bins==4)*(mu_bins==0)
        ana_mut[i][j] = bs_data[4][mu_t_bin]
        u1_mut[i][j] = bs_data[0][mu_t_bin]
        tri_mut[i][j] = bs_data[6][mu_t_bin]
    return u1_mut,ana_mut,tri_mut

In [ ]:
## here bs_val = data[mm_bin][tt_bin],  u1_val=u1_mut[mm_bin][tt_bin]

def rebin_u1(u1_val,bs_val,tri_val,NB):
    binsiz = np.log10(np.max(uavg)/np.min(uavg)) / (1.0 * NB)
    no = np.zeros(NB, dtype=int)
    bs_sum = np.zeros(NB, dtype=float)
    tri_sum= np.zeros(NB, dtype=float)
    usum = np.zeros(NB, dtype=float)
    for i in range(len(bs_val)):
        x_val = uavg[int(u1_val[i])]
        y_val = bs_val[i]
        wt_val= tri_val[i]
        tmp = int(np.floor(np.log10(x_val/np.min(uavg)) / binsiz))
        tmp = tmp if tmp < NB else tmp - 1
        no[tmp] += 1
        bs_sum[tmp] += (y_val*wt_val)
        tri_sum[tmp] += wt_val
        usum[tmp] += x_val

    usum /=  no
    bs_sum /= tri_sum
    
    return usum*dU,bs_sum

In [ ]:
def draw_tri():
    triangle = patches.RegularPolygon((0.22, 0.3), numVertices=3, radius=0.13, orientation=0, 
                                  edgecolor='black', facecolor='none', transform=plt.gca().transAxes)

    plt.gca().add_patch(triangle)

    vertices = triangle.get_verts()

    trans = plt.gca().transAxes.inverted()
    vertices = trans.transform(vertices)

    plt.plot([vertices[0, 0], vertices[1, 0]], [vertices[0, 1], vertices[1, 1]], color='blue', transform=plt.gca().transAxes,ls='--')
    plt.plot([vertices[1, 0], vertices[2, 0]], [vertices[1, 1], vertices[2, 1]], color='orange', transform=plt.gca().transAxes,ls='--')
    plt.plot([vertices[2, 0], vertices[0, 0]], [vertices[2, 1], vertices[0, 1]], color='red', transform=plt.gca().transAxes,ls='--')


    midpoints=np.array([[0.14, 0.35    ],
     [0.22 , 0.2],
     [0.3, 0.35    ]])
    for midpoint in midpoints:
        plt.text(midpoint[0], midpoint[1], r'$\ell$', fontsize=14, ha='center', va='center', transform=plt.gca().transAxes)

    box_x = 0.06  # x-coordinate of the lower-left corner
    box_y = 0.16  # y-coordinate of the lower-left corner
    box_width = 0.33  # width of the box
    box_height = 0.33  # height of the box

    box = patches.Rectangle((box_x, box_y), box_width, box_height, linewidth=1, 
                            edgecolor='black', facecolor='none', transform=plt.gca().transAxes)

    plt.gca().add_patch(box)

In [ ]:
# loc='/home/sukhdeep/Documents/Research/Bispectrum/Visibility_Bispectrum/Visibility_Correlation/asif_da/4_fbe/results_final'

# num_bins = 5     ## binning in mu,t 
# NB = 10          ## rebinning in u1

# # bs_rlzns = []
# # bs_rlzns = np.empty(500,dtype=object)
# storage = [[[] for _ in range(num_bins)] for _ in range(num_bins)]
# u1_mut,ana_mut,tri_mut = binned_ana_data(num_bins)
# for rlzn in range(1,501):
#     data = binned_data(rlzn,num_bins)
    
# #     final_u1 = np.empty((num_bins,num_bins), dtype=object)
# #     final_bs = np.empty((num_bins,num_bins), dtype=object)
#     for i, j in itertools.product(range(num_bins), range(num_bins)):
#         x1,y1 = rebin_u1(u1_mut[i][j],data[i][j],tri_mut[i][j],NB)
# #         final_u1[i][j] = x1
# #         final_bs[i][j] = y1
#         storage[i][j].append(y1)


In [ ]:
loc='/home/sukhdeep/Documents/Research/Bispectrum/Visibility_Bispectrum/Visibility_Correlation/asif_da/4_fbe/results_final'

num_bins = 5     ## binning in mu,t 
NB = 10          ## rebinning in u1

storage = [[[] for _ in range(num_bins)] for _ in range(num_bins)]
u1_mut,ana_mut,tri_mut = binned_ana_data(num_bins)
for rlzn in range(1,501):
    data = binned_data(rlzn,num_bins)
    
    for i, j in itertools.product(range(num_bins), range(num_bins)):
        x1,y1 = rebin_u1(u1_mut[i][j],data[i][j],tri_mut[i][j],NB)
        storage[i][j].append(y1)


In [ ]:
mean_final_bs = np.empty((num_bins, num_bins), dtype=object)
std_final_bs = np.empty((num_bins, num_bins), dtype=object)

final_u1 = np.empty((num_bins, num_bins), dtype=object)
final_ana = np.empty((num_bins, num_bins), dtype=object)

for i, j in itertools.product(range(num_bins), range(num_bins)):
    stacked_elements = np.array(storage[i][j])
    mean_final_bs[i][j] = np.mean(stacked_elements, axis=0)
    std_final_bs[i][j] = np.std(stacked_elements, axis=0)
    
    x1, y1 = rebin_u1(u1_mut[i][j],ana_mut[i][j],tri_mut[i][j],NB)
    final_u1[i][j]  = x1
    final_ana[i][j] = y1
    

In [ ]:
plt.rc('text',usetex=True)
from matplotlib import rc
rc('font', **{'family': 'serif', 'serif': ['Computer Modern']})
rc('text', usetex=True)

In [ ]:
mm=0
tt=4
tpi=2*np.pi

a1 = final_u1[mm][tt]
b1 = mean_final_bs[mm][tt]*beam_norm_fact/Q_cube
c1 = std_final_bs[mm][tt]*beam_norm_fact/(np.sqrt(499)*Q_cube)

plt.errorbar(tpi*a1[1:],b1[1:],yerr=c1[1:] , fmt='o',markersize=4.5, elinewidth=1.5, capsize=3,markeredgecolor='black', color='red', ecolor='black',label='Estimated')
plt.plot(tpi*a1[1:],final_ana[mm][tt][1:]*(ff)/Q_cube,linestyle="--",color='olive',label='Predicted')
# plt.yscale("symlog",linthresh=.01)
plt.text(65, 3e66, 'Equilateral', ha='left', va='bottom',fontsize=15)

plt.yscale("log")
plt.xscale("log")

plt.xlabel("$\ell${ rad}$^{-1}$",size=20)
plt.ylabel("$B(\ell)$",size=20)

plt.legend(fontsize=15)

plt.xticks(fontsize=15)
plt.yticks(fontsize=15)


draw_tri()

# plt.savefig('plots/equi_bs_v2.pdf', dpi=300, bbox_inches='tight')


In [ ]:
print(tpi*a1[1:])
print(a1[1:])

In [ ]:
# (b1[1:]-(final_ana[mm][tt][1:]*(ff)/Q_cube))  /  (final_ana[mm][tt][1:]*(ff)/Q_cube)
(b1[1:]-(final_ana[mm][tt][1:]*(ff)/Q_cube))  /  (c1[1:])

In [ ]:
tpi*350

In [ ]:
## Refinning some data

mean_final_bs[0][3][5]=np.nan

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Assuming mean_final_data, std_final_data, final_u1, and final_ana are defined as 2D arrays with shape (5, 5)
num_bins = 5  # As the subplots grid is 5x5

# Create a 5x5 subplot figure
fig, axs = plt.subplots(num_bins, num_bins, figsize=(15, 15), sharex=True, sharey=False)

lbl_fz = 20
# Iterate over each subplot to plot data
for i in range(num_bins):
    for j in range(num_bins):
        
        ax = axs[i,j]
        
        mm = j
        tt = abs(i-(num_bins-1))
        
        a1 = final_u1[mm][tt]
        b1 = mean_final_bs[mm][tt]*beam_norm_fact/Q_cube
        c1 = std_final_bs[mm][tt]*beam_norm_fact/(np.sqrt(499)*Q_cube)
        th_bs=final_ana[mm][tt]*(ff)/Q_cube
    

        # Plot mean data with error bars
        ax.errorbar(tpi*a1, b1, yerr=c1, fmt='o',markersize=6, elinewidth=1.6, capsize=3,markeredgecolor='black', color='red', ecolor='black',label='Estimated')

        
        if (j==3 and (i==0 or i==3)):
            valid_indices = ~np.isnan(a1) & ~np.isnan(th_bs)
            a1 = a1[valid_indices]
            th_bs = th_bs[valid_indices]
        
        # Plot the dashed line (final_u1, final_ana)
        ax.plot(tpi*a1, th_bs, linestyle='--',color='olive',linewidth=2, label='Predicted')
        
        ax.set_yscale("log")
        ax.set_xscale("log")
        
        ax.set_xlim(30,2500)
        
        ax.tick_params(axis='y', which='both', direction='in', labelsize=14)
        ax.tick_params(axis='x',labelsize=17)  
        
        ax.tick_params(axis='both', which='major', length=8, width=2)
        ax.tick_params(axis='both', which='minor', length=4, width=1)


        # Set axis labels
        if (i == num_bins - 1):
            ax.set_xlabel(r'$\ell_1$ rad$^{-1}$',size=lbl_fz)
        if j == 0:
            ax.set_ylabel(r'$B(\ell_1, \mu, t)$',size=lbl_fz)
            
        if (i==1 and j==0) or(i==2 and j==1) or (j==2 and i==3) or (j==3 and i==4):
            ax.set_xlabel(r'$\ell_1$ rad$^{-1}$',size=lbl_fz)
            ax.set_ylabel(r'$B(\ell_1, \mu, t)$',size=lbl_fz)
            ax.tick_params(labelbottom=True)

        # Optionally, set limits (if needed)
#         ax.set_xlim([min(final_u1[i][j]), max(final_u1[i][j])])
#         ax.set_ylim([min(mean_final_bs[i][j] - std_final_bs[i][j]), max(mean_final_bs[i][j] + std_final_bs[i][j])])


# Adjust layout to avoid overlap
plt.tight_layout()

# Add a legend to the first subplot (if you want it globally)
handles, labels = axs[0, 0].get_legend_handles_labels()
# fig.legend(handles, labels, loc='center left', ncol=1,fontsize=20)
fig.legend(handles, labels, loc='center', bbox_to_anchor=(0.1, 0.2), fontsize=30)


axs[2][0].axis('off')
axs[3][0].axis('off')
axs[3][1].axis('off')
axs[4][0].axis('off')
axs[4][1].axis('off')
axs[4][2].axis('off')


for i in range(num_bins):
    for j in range(num_bins):
        mm = j
        tt = abs(i-(num_bins-1))
        if (((mm*tt>.5) or i==0 or i==1 or j==3 or j==4)):
            if not (i == 3 and j == 1):
                ax = axs[i,j]
                ax.text(0.1, 0.25, f'$\mu = {0.55 + 0.1*mm:.2f}$', ha='left', va='bottom',transform=ax.transAxes,fontsize=16)
                ax.text(0.1, 0.1, f'$t = {0.55 + 0.1*tt:.2f}$', ha='left', va='bottom',transform=ax.transAxes,fontsize=16)


fig.subplots_adjust(left=-.01, bottom=None, right=1.08, top=None, wspace=0.18, hspace=.03) 
# Show or save the plot
# plt.show()  # Or use plt.savefig('your_filename.png') to save the plot
# plt.savefig('plots/all_tri_LPbs_Q3.pdf', dpi=300, bbox_inches='tight')


In [ ]:
a1

In [ ]:
a1[:-1]

# $\mu-t$ plots

In [ ]:
dU*np.max(uavg[:-3])

In [ ]:
np.logspace(np.log10(dU*np.min(uavg)), np.log10(dU*np.max(uavg)), 10 + 1)

In [ ]:
np.logspace(np.log10(dU*np.min(uavg)), np.log10(dU*np.max(uavg)), NB + 1)

In [ ]:
uval = dU*uavg

def get_bin_number_u1(values, NB):
    values = np.asarray(values)
#     binsiz = np.log10(np.max(uavg)/np.min(uavg)) / (1.0 * NB)
#     bin_numbers = int(np.floor(np.log10(values/np.min(uavg)) / binsiz))
    bin_edges = np.logspace(np.log10(dU*np.min(uavg)), np.log10(dU*np.max(uavg)), NB + 1)
    bin_numbers = np.digitize(values, bin_edges, right=False) - 1
    bin_numbers = np.where(bin_numbers == NB, NB - 1, bin_numbers)
    return bin_numbers

def binned_data_u1(rlzn,NB):
    bs_u1 = np.empty(NB, dtype=object)
    bs_data = stack_data(rlzn)
    xx = [dU*uavg[int(i)] for i in bs_data[0]]
    u1_bins  = get_bin_number_u1(xx,NB)
    for i in range(NB):
        uu11_bin = np.where((u1_bins == i))   # (t_bins==4)*(mu_bins==0)
        bs_u1[i] = bs_data[3][uu11_bin]
    return bs_u1


def binned_ana_data_u1(NB):
    mu_u1  = [[] for _ in range(NB)] 
    t_u1   = [[] for _ in range(NB)]
    tri_u1 = [[] for _ in range(NB)] 
    ana_u1 = [[] for _ in range(NB)] 
    
    bs_data = stack_data(1)
    xx = [dU*uavg[int(i)] for i in bs_data[0]]
    u1_bins  = get_bin_number_u1(xx,NB)
    for i in range(NB):
        uu11_bin  = np.where((u1_bins == i))
        ana_u1[i] = bs_data[4][uu11_bin]
        mu_u1[i]  = bs_data[2][uu11_bin]
        t_u1[i]   = bs_data[1][uu11_bin]
        tri_u1[i] = bs_data[6][uu11_bin]
    return mu_u1,t_u1,ana_u1,tri_u1

## here bs_val = data[mm_bin][tt_bin],  u1_val=u1_mut[mm_bin][tt_bin]
def find_mut_bin_lin(val,num_bins):
    binsiz = 0.5 / (1.0 * num_bins)
    tmp = int(np.floor((val-0.5) / binsiz))
    tmp = tmp if tmp < num_bins else tmp - 1
    return tmp

def rebin_mut(mu_val,t_val,bs_val,tri_val,num_bins):
#     no     = np.zeros((num_bins,num_bins), dtype=int)
    bs_sum = np.zeros((num_bins,num_bins), dtype=float)
    tri_sum= np.zeros((num_bins,num_bins), dtype=float)
    for i in range(len(bs_val)):
        y_val = bs_val[i]
        wt_val= tri_val[i]
        t_bin = find_mut_bin_lin(t_val[i],num_bins)
        mu_bin= find_mut_bin_lin(mu_val[i],num_bins)

#         no[mu_bin][t_bin] += 1
        bs_sum[mu_bin][t_bin] += (y_val*wt_val)
        tri_sum[mu_bin][t_bin] += wt_val
        
    bs_sum /= tri_sum
    
    return bs_sum

In [ ]:
# mu_u1,t_u1,ana_u1,tri_u1=binned_ana_data_u1(NB=5)
# plt.scatter(mu_u1[4],t_u1[4])

In [ ]:
loc='/home/sukhdeep/Documents/Research/Bispectrum/Visibility_Bispectrum/Visibility_Correlation/asif_da/4_fbe/results_final'

num_bins = 5     ## rebinning in mu,t 
NB = 10          ## binning in u1

storage = [[] for _ in range(NB)] 

mu_u1,t_u1,ana_u1,tri_u1 = binned_ana_data_u1(NB)
for rlzn in range(1,501):
    data = binned_data_u1(rlzn,NB)
    
    for i in range(NB):
        y1 = rebin_mut(mu_u1[i],t_u1[i],data[i],tri_u1[i],num_bins)
        storage[i].append(y1)
  

In [ ]:
mean_bs_u1 = np.empty(NB, dtype=object)
std_bs_u1  = np.empty(NB, dtype=object)

final_ana_u1 = np.empty(NB, dtype=object)

for i in range(NB):
    stacked_elements = np.array(storage[i])
    mean_bs_u1[i]    = np.mean(stacked_elements, axis=0)
    std_bs_u1[i]     = np.std(stacked_elements, axis=0)
    
    y1 = rebin_mut(mu_u1[i],t_u1[i],ana_u1[i],tri_u1[i],num_bins)
    final_ana_u1[i] = y1
        

In [ ]:
i=7
rebin_mut(mu_u1[i],t_u1[i],tri_u1[i],tri_u1[i],num_bins)

In [ ]:
# mm = 4
# tt = 2
# uu1= 8
# print(final_ana_u1[uu1][mm][tt],final_ana[mm][tt][uu1])
# print(mean_bs_u1[uu1][mm][tt],mean_final_bs[mm][tt][uu1])

# Heat plots

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import cmath
import scipy.interpolate as itp
from matplotlib.colors import LogNorm,SymLogNorm
import matplotlib as mpl
from matplotlib import cm
from matplotlib.ticker import MultipleLocator
import pandas as pd
import time

# import scipy.special as sci
import matplotlib.ticker as ticker
import matplotlib.colors as colors

In [ ]:
# fig, ax = plt.subplots(nrows=4, ncols=4, figsize=(40, 24),sharey=False)
plt.rc('text',usetex=True)
from matplotlib import rc
rc('font', **{'family': 'serif', 'serif': ['Computer Modern']})
rc('text', usetex=True)
color=['#1f77b4','darkgreen','navy','olive','#d62728','#ff7f0e']

# pblsh heatplots

In [ ]:
plt.rc('text',usetex=True)
xll = .5;  xul = 1;  yll = .5;  yul = 1

ncw=plt.cm.get_cmap('viridis_r')  

colors_cstmz = ['midnightblue', 'forestgreen', 'yellow', 'darkred'] #goldenrod, darkorange
bounds = [0, 0.05, .1, .15, .2]
ncw_cstmz = plt.cm.colors.ListedColormap(colors_cstmz)
norm_cstmz = plt.cm.colors.BoundaryNorm(bounds, ncw_cstmz.N)
# ###========================l = 2=================================


MuT_bin=12
fig2, axes2 = plt.subplots(nrows=1, ncols=3, figsize=(7*3,7*1), gridspec_kw={'width_ratios': [1, 1,1]},sharey=True)


ii=7#7

fft_mpl = (mean_bs_u1[ii].T)*beam_norm_fact/Q_cube
ana_mpl = (final_ana_u1[ii].T)*ff/Q_cube
sd_mpl  = (std_bs_u1[ii].T)*beam_norm_fact/(np.sqrt(499)*Q_cube)

deviation = (np.abs(fft_mpl-ana_mpl))/ana_mpl

mini=np.min([np.nanmin(ana_mpl),np.nanmin(fft_mpl)])
maxi=np.max([np.nanmax(ana_mpl),np.nanmax(fft_mpl)])

# clr_norm=mpl.colors.SymLogNorm(.01,vmin=mini,vmax=maxi)
clr_norm= mpl.colors.Normalize()   ##mpl.colors.LogNorm(2e8) #

cp    =  axes2[0].imshow(fft_mpl/1e67, extent=[xll, xul, yll, yul], origin='lower', cmap=ncw,norm=clr_norm)#
axes2[1].imshow(ana_mpl/1e67, extent=[xll, xul, yll, yul], origin='lower', cmap=ncw,norm=clr_norm)#

#     clr_norm=mpl.colors.SymLogNorm(.001)
#     cp_fd =  axes2[ii,1].imshow((fft_mpl-ana_mpl)/ana_mpl, extent=[xll, xul, yll, yul], origin='lower', cmap=plt.cm.get_cmap('RdBu'),norm=clr_norm)


clr_norm=mpl.colors.SymLogNorm(.001)

cp_err=axes2[2].imshow(deviation, extent=[xll, xul, yll, yul], origin='lower', cmap=ncw_cstmz,norm=norm_cstmz )

lbls=['Estimated','Predicted',r'$\Delta$']
for lz in range(3):   
    axes2[lz].tick_params(axis='y', which='both', direction='out' , length=8, width=1.5,labelsize=20)
    axes2[lz].tick_params(axis='x',labelsize=20, length=8, width=1.5)  
    axes2[lz].set_xlabel(r'$\mu$',size=30)
    axes2[lz].text(0.08, 0.08, lbls[lz], ha='left', va='bottom',transform=axes2[lz].transAxes,fontsize=25)

axes2[0].set_ylabel(r'$t$',size=30)

# axes2[1].tick_params(axis='both', which='major', length=8, width=2)
# ax.tick_params(axis='both', which='minor', length=4, width=1)
 #####################################   Add colorbars #############################

cax = fig2.add_axes([0.12,1.08,.483,0.07])  ## right, up, width, height , right=-.08
cbar=fig2.colorbar(cp,cax=cax, label=r'$B_(\ell_1,\mu,t)$',extend='both',pad=1,orientation="horizontal")
cbar.set_label(label=r'',size=30,weight='bold',labelpad=1,loc='center')
cbar.ax.tick_params(labelsize=27,labelbottom=True,labelleft=False,labelright=False,labeltop=False,right=False,left=False,bottom=True,top=False)
# cbar.ax.xaxis.set_minor_locator(ticker.SymmetricalLogLocator(linthresh=.01,base=10,subs=[2, 3, 4, 5, 6, 7, 8, 9]))
cbar.ax.tick_params(which='major', length=10,width=2.5)
cbar.ax.tick_params(which='minor', length=6,width=1.2,left=True,right=False)
cbar.set_label(label=r'$B(\ell_1,\mu,t)$',size=30,weight='bold',labelpad=40,loc='center',va='bottom')
axes2[1].text(.938, 1.155, r'$\times 10^{67}$', fontsize=25)
  
    

up_param=[.92,.9,.442,.21] 
cax = fig2.add_axes([0.66,1.08,.23,0.07])  ## right, up, width, height, right=0.283, width=.255
cbar=fig2.colorbar(cp_err,cax=cax, label=r'$B_0^0$',extend='both',pad=1,orientation="horizontal")
cbar.set_label(label=r'',size=30,weight='bold',labelpad=1,loc='center')
cbar.ax.tick_params(labelsize=25,rotation=0)
#     cbar.ax.xaxis.set_minor_locator(ticker.SymmetricalLogLocator(linthresh=.0001,base=10,subs=[2, 3, 4, 5, 6, 7, 8, 9]))
cbar.ax.tick_params(which='major', length=10,width=2.5)
cbar.ax.tick_params(which='minor', length=6,width=1.2)
cbar.set_label(label=r'$\Delta$',size=30,weight='bold',labelpad=15,loc='center')

# ####################################################################################
    
# fnt_sz=24
# for jz in range(4):
#     labelkeyy=1 if jz==3 else 0
#     axes2[jz][0].xaxis.set_tick_params(which='both', top=True, labeltop=False,
#                              bottom=True, labelbottom=labelkeyy, labelsize=fnt_sz)
#     axes2[jz][0].yaxis.set_tick_params(which='both', top=True, labeltop=False,
#                              bottom=True, labelbottom=1, labelsize=fnt_sz)

# #     axes2[jz][1].xaxis.set_tick_params(which='both', top=True, labeltop=False,
# #                              bottom=True, labelbottom=labelkeyy, labelsize=fnt_sz)
# #     axes2[jz][1].yaxis.set_tick_params(which='both', top=True, labeltop=False,
# #                              bottom=0, labelbottom=0, labelsize=fnt_sz)

#     axes2[jz][1].xaxis.set_tick_params(which='both', top=True, labeltop=False,
#                              bottom=True, labelbottom=labelkeyy, labelsize=fnt_sz)
#     axes2[jz][1].yaxis.set_tick_params(which='both', top=True, labeltop=False,
#                              bottom=0, labelbottom=0, labelsize=fnt_sz)
for jz in range(3): 
    axes2[jz].set_xticks([.507,.6,.7,.8,.9,.993])
    axes2[jz].set_xticklabels(['0.5','0.6','0.7','0.8','0.9','1'])
    

    

x=np.linspace(.5,1,50)
y=1/(2*x)
for lz in range(3):
        axes2[lz].plot(x,y,"--",color="black",linewidth=1.7)

fig2.subplots_adjust(left=None, bottom=None, right=None, top=None, wspace=0.06, hspace=None) 
# fig2.savefig('plots/u1_7.pdf', bbox_inches='tight',dpi = 500) 
# plt.savefig('plots/all_tri_LPbs.pdf', dpi=300, bbox_inches='tight')


In [ ]:
mini,maxi

In [ ]:
deviation*100

In [ ]:
_=plt.hist(deviation.flatten(),50)

In [ ]:
_=plt.hist(deviation.flatten(),50)

In [ ]:
_=plt.hist(deviation.flatten(),50)

# SNR

In [ ]:
# ell_val = tpi*np.array([  7.9756327 ,  12.98888754,  18.06724959,
#                 28.27901381,  47.24068167,  77.02319919, 110.42303105,
#                188.42196951])

In [ ]:
xx=np.logspace(np.log10(dU*np.min(uavg)), np.log10(dU*np.max(uavg)), 10 + 1)
xx_low=xx[0:10]
xx_high=xx[1:11]
xx_avg=(2/3)*(np.array(xx_high)**3-np.array(xx_low)**3)/(np.array(xx_high)**2-np.array(xx_low)**2)

In [ ]:
mid=(xx_low+xx_high)/2

In [ ]:
tpi=2*np.pi
ell_val=tpi*mid

### plot

In [ ]:
plt.rc('text',usetex=True)
xll = .5;  xul = 1;  yll = .5;  yul = 1

nrm_val=[.1,.6,.01,.01,0.01,0.01,0.01,0.01,0.01,0.01,0.01,0.01,0.01,0.01,0.01,0.01]
# clr_norm=mpl.colors.SymLogNorm(10**(-1),vmin=0,vmax=10)

ncw=plt.cm.get_cmap('viridis_r')  

colors_cstmz = ['midnightblue', 'forestgreen', 'yellow', 'darkorange','darkred'] #goldenrod, darkorange
bounds = [0, 1, 2, 3, 4, 5]
ncw_cstmz = plt.cm.colors.ListedColormap(colors_cstmz)
norm_cstmz = plt.cm.colors.BoundaryNorm(bounds, ncw_cstmz.N)
# ###========================l = 2=================================


MuT_bin=12
fig2, axes2 = plt.subplots(nrows=2, ncols=4, figsize=(7*4,7*2), gridspec_kw={'width_ratios': [1, 1,1,1]},sharex=True,sharey=True)


for ii in range(8):
    
    i = ii//4
    j = ii if ii<4 else 7-ii
    fft_mpl = (mean_bs_u1[ii+1].T)*beam_norm_fact
    ana_mpl = (final_ana_u1[ii+1].T)*ff
    sd_mpl  = (std_bs_u1[ii+1].T)*beam_norm_fact/np.sqrt(499)

    deviation = (np.abs(fft_mpl-ana_mpl))/sd_mpl

    cp_err=axes2[i][j].imshow(deviation, extent=[xll, xul, yll, yul], origin='lower', cmap=ncw_cstmz,norm=norm_cstmz )

    axes2[i][j].text(0.07, 0.076, r'$\ell_1=%d$'%(ell_val[ii+1]), ha='left', va='bottom',transform=axes2[i][j].transAxes,fontsize=29)

    print(np.nanmax(deviation))
    
# lbls=['Estimate','Analytical',r'$\Delta_{f}$']
for jz in [0,1]:
    for lz in range(4):  
        axes2[jz][lz].tick_params(axis='y', which='both', direction='out' , length=8, width=1.5,labelsize=21)
        axes2[jz][lz].tick_params(axis='x',labelsize=21, length=8, width=1.5)  
        if jz==1:
            axes2[jz][lz].set_xlabel(r'$\mu$',size=35)
        if lz==0:
            axes2[jz][lz].set_ylabel(r'$t$',size=35)
#     axes2[lz].text(0.08, 0.08, lbls[lz], ha='left', va='bottom',transform=axes2[lz].transAxes,fontsize=25)




 #####################################   Add colorbars #############################

cax = fig2.add_axes([.918,.118,0.019,.76])  ## right, up, width, height, right=0.283, width=.255
cbar=fig2.colorbar(cp_err,cax=cax, label=r'$B_0^0$',extend='both',pad=1,orientation="vertical")
cbar.set_label(label=r'',size=30,weight='bold',labelpad=1,loc='center')
cbar.ax.tick_params(labelsize=25,rotation=0)
#     cbar.ax.xaxis.set_minor_locator(ticker.SymmetricalLogLocator(linthresh=.0001,base=10,subs=[2, 3, 4, 5, 6, 7, 8, 9]))
cbar.ax.tick_params(which='major', length=10,width=2.5)
cbar.ax.tick_params(which='minor', length=6,width=1.2)
cbar.set_label(label=r'$\Delta_\sigma$',size=35,weight='bold',labelpad=8,loc='center')

# ####################################################################################
    
# fnt_sz=24
for jz in range(4):
#     labelkeyy=1 if jz==3 else 0
#     axes2[jz][0].xaxis.set_tick_params(which='both', top=True, labeltop=False,
#                              bottom=True, labelbottom=labelkeyy, labelsize=fnt_sz)
#     axes2[jz][0].yaxis.set_tick_params(which='both', top=True, labeltop=False,
#                              bottom=True, labelbottom=1, labelsize=fnt_sz)

# #     axes2[jz][1].xaxis.set_tick_params(which='both', top=True, labeltop=False,
# #                              bottom=True, labelbottom=labelkeyy, labelsize=fnt_sz)
# #     axes2[jz][1].yaxis.set_tick_params(which='both', top=True, labeltop=False,
# #                              bottom=0, labelbottom=0, labelsize=fnt_sz)

#     axes2[jz][1].xaxis.set_tick_params(which='both', top=True, labeltop=False,
#                              bottom=True, labelbottom=labelkeyy, labelsize=fnt_sz)
#     axes2[jz][1].yaxis.set_tick_params(which='both', top=True, labeltop=False,
#                              bottom=0, labelbottom=0, labelsize=fnt_sz)
    axes2[1][jz].set_xticks([.508,.6,.7,.8,.9,0.994])
    axes2[1][jz].set_xticklabels(['0.5','0.6','0.7','0.8','0.9','1'])
    axes2[1][jz].set_yticks([.507,.6,.7,.8,.9,.993])
    axes2[1][jz].set_yticklabels(['0.5','0.6','0.7','0.8','0.9','1'])
#     axes2[jz].set_yticks([.507,.6,.7,.8,.9,.993])
#     axes2[jz].set_yticklabels(['0.5','0.6','0.7','0.8','0.9','1'])
    
    
    
# axes2[3][0].set_xlabel("$\mu$",fontsize=45)
# # axes2[3][1].set_xlabel("$\mu$",fontsize=45)
# axes2[3][1].set_xlabel("$\mu$",fontsize=45)
# for jz in range(4):
#     axes2[jz][0].set_ylabel("$t$",fontsize=45)

    


x=np.linspace(.5,1,50)
y=1/(2*x)
for jz in [0,1]:
    for lz in range(4):
        axes2[jz][lz].plot(x,y,"--",color="black",linewidth=1.7)
        
fig2.subplots_adjust(left=None, bottom=None, right=None, top=None, wspace=0.03, hspace=0) 
             

# fig2.savefig('plots/snr.pdf', bbox_inches='tight',dpi = 500) 

# $N_{tri}$ heatplot

In [ ]:
ntri_arr

In [ ]:
fig, ax = plt.subplots(nrows=1, ncols=1, figsize=(6,6))
from matplotlib import rc
rc('font', **{'family': 'serif', 'serif': ['Computer Modern']})
rc('text', usetex=True)
# fig.subplots_adjust(left=.245, bottom=None, right=None, top=None, wspace=0, hspace=0.02) 
ncw=plt.cm.get_cmap('YlGnBu')  #turbo,RdYlBu YlGnBu

MuT_bin=12
xll = .5;  xul = 1;  yll = .5;  yul = 1

#--------------------------------------------------------------------

ii=7#7
xx=np.array([[     0.,      0.,      0.,  18956.,  50158.],
       [     0.,      0.,  29964.,  28788.,  31492.],
       [     0.,  24454.,  31202.,  26738.,  44722.],
       [ 41094.,  60412.,  60822.,  51828.,  43130.],
       [131612., 144912., 255392., 411751., 601267.]])

ntri_arr = xx.T

ntri_arr[ntri_arr==0]=np.nan

mini=np.min([np.nanmin(ntri_arr)])
maxi=np.max([np.nanmax(ntri_arr)])


clr_norm= mpl.colors.LogNorm()#mpl.colors.Normalize()   ##mpl.colors.LogNorm(2e8) #

cp=ax.imshow(ntri_arr/1e4, extent=[xll, xul, yll, yul], origin='lower', cmap=ncw,norm=clr_norm)#

ax.tick_params(axis='y', which='both', direction='out' , length=8, width=1.5,labelsize=15)
ax.tick_params(axis='x',labelsize=15, length=8, width=1.5)  
# cax = fig.add_axes([1.1,.1,.03,0.78])  ## right, up, width, height
cax = fig.add_axes([.13, -.05, .786, 0.025]) 
cbar=fig.colorbar(cp,cax=cax, label=r'$B_2^0$',extend='both',pad=1,orientation="horizontal")
cbar.set_label(label=r'$N_{\rm tri}$',size=15,weight='bold',labelpad=9,loc='center')
cbar.ax.tick_params(labelsize=12,direction='out')
# cbar.ax.yaxis.set_minor_locator(ticker.SymmetricalLogLocator(linthresh=.1,base=10,subs=[2, 3, 4, 5, 6, 7, 8, 9]))

cbar.ax.tick_params(which='major', length=8,width=2)
cbar.ax.tick_params(which='minor', length=5,width=1)


x=np.linspace(.5,1,50)
y=1/(2*x)
ax.plot(x,y,"--",color="black",linewidth=1.1)
ax.set_xlabel("$\mu$",fontsize=20)
ax.set_ylabel("$t$",fontsize=20)
ax.text(.53, .55, r'$\ell_1 = 762 ~{\rm rad}^{-1}$', fontsize=18, color='black')





ax.text(0.41, .99, 'Equilateral', fontsize=17, rotation=45)
ax.text(0.99, 0.99, 'Squeezed', fontsize=17, rotation=-45)
ax.text(.983, 0.43, 'Stretched', fontsize=17, rotation=45)


ax.text(.71, 1.013, 'L Isosceles', fontsize=17, rotation=0)
ax.text(1.01, 0.7, 'Linear', fontsize=17, rotation=90)
# ax.text(.68, 0.89, 'Acute', fontsize=22, rotation=0)
# ax.text(.85, 0.66, 'Obtuse', fontsize=22, rotation=0)
# ax.text(.74, 0.775, 'Right angle', fontsize=20, rotation=45)
ax.text(.65, 0.64, 'S Isosceles', fontsize=17, rotation=-47)

ax.annotate('', xy=(1.04, 1), xycoords='axes fraction', xytext=(1.04, .55),arrowprops=dict(arrowstyle="->", color='black',lw=1.3))
ax.annotate('', xy=(1.04, .37), xycoords='axes fraction', xytext=(1.04, 0),arrowprops=dict(arrowstyle="<-", color='black',lw=1.3))


ax.annotate('', xy=(.68, 1.04), xycoords='axes fraction', xytext=(1, 1.04),arrowprops=dict(arrowstyle="<-", color='black',lw=1.15))
ax.annotate('', xy=(.41, 1.04), xycoords='axes fraction', xytext=(0, 1.04),arrowprops=dict(arrowstyle="<-", color='black',lw=1.15))

x=np.arange(.748,.9,.001)
y=1/(2*x)-.038
ax.plot(x,y,'-',linewidth=1.3,color='black')
ax.annotate('', xy=(.7, .1), xycoords='axes fraction', xytext=(.82, .022),arrowprops=dict(arrowstyle="<-", color='black',lw=1.))


x=np.arange(.515,.649,.001)
y=1/(2*x)-.038
ax.plot(x,y,'-',linewidth=1.3,color='black')
ax.annotate('', xy=(.05, .828), xycoords='axes fraction', xytext=(0.012, .9),arrowprops=dict(arrowstyle="<-", color='black',lw=1.1))


ax.text(1.015, .397, r'$\times 10^4$', fontsize=14)

x1=np.arange(.707,1,.001)
ax.plot(x1,x1,'-.',linewidth=1.1,color='blue')
# fig.savefig('plots/ntri.pdf', bbox_inches='tight',dpi=300)  

# Extra

In [ ]:
plt.rc('text',usetex=True)
xll = .5;  xul = 1;  yll = .5;  yul = 1

nrm_val=[.1,.6,.01,.01,0.01,0.01,0.01,0.01,0.01,0.01,0.01,0.01,0.01,0.01,0.01,0.01]
# clr_norm=mpl.colors.SymLogNorm(10**(-1),vmin=0,vmax=10)

ncw=plt.cm.get_cmap('viridis_r')  

colors_cstmz = ['midnightblue', 'forestgreen', 'yellow', 'darkred'] #goldenrod, darkorange
bounds = [0, 1, 3, 5, 7]
ncw_cstmz = plt.cm.colors.ListedColormap(colors_cstmz)
norm_cstmz = plt.cm.colors.BoundaryNorm(bounds, ncw_cstmz.N)
# ###========================l = 2=================================


MuT_bin=12
fig2, axes2 = plt.subplots(nrows=9, ncols=4, figsize=(7*9,7*4), gridspec_kw={'width_ratios': [1.2, 1.2,.05,1.2]})
# fig2.subplots_adjust(left=.33, bottom=None, right=None, top=None, wspace=0.15, hspace=None)

# up_param=[.6922,.498,.304,.1085]
up_param=[.6922,.5,.4,.3,.2,.1,.07,.04,.01]
for ii in range(9):
    fft_mpl = (mean_bs_u1[ii+1].T)*beam_norm_fact
    ana_mpl = (final_ana_u1[ii+1].T)*ff
    sd_mpl  = (std_bs_u1[ii+1].T)*beam_norm_fact/np.sqrt(499)
    
    deviation = (np.abs(fft_mpl-ana_mpl))/sd_mpl
    
    mini=np.min([np.nanmin(ana_mpl),np.nanmin(fft_mpl)])
    maxi=np.max([np.nanmax(ana_mpl),np.nanmax(fft_mpl)])
    
    clr_norm=mpl.colors.SymLogNorm(nrm_val[ii],vmin=mini,vmax=maxi)
#     clr_norm= mpl.colors.Normalize()   ##mpl.colors.LogNorm(2e8) #

    cp    =  axes2[ii,0].imshow(fft_mpl, extent=[xll, xul, yll, yul], origin='lower', cmap=ncw,norm=clr_norm)#
    
#     clr_norm=mpl.colors.SymLogNorm(.001)
#     cp_fd =  axes2[ii,1].imshow((fft_mpl-ana_mpl)/ana_mpl, extent=[xll, xul, yll, yul], origin='lower', cmap=plt.cm.get_cmap('RdBu'),norm=clr_norm)
    axes2[ii,2].axis('off')
    
    cp_ana=axes2[ii,1].imshow(ana_mpl, extent=[xll, xul, yll, yul], origin='lower', cmap=ncw,norm=clr_norm )
    
    clr_norm=mpl.colors.SymLogNorm(.001)
    cp_err=axes2[ii,3].imshow(deviation, extent=[xll, xul, yll, yul], origin='lower', cmap=ncw_cstmz,norm=norm_cstmz )
    
       
    
 #####################################   Add colorbars #############################

    cax = fig2.add_axes([0.08,up_param[ii],.017,0.1885])  ## right, up, width, height , right=-.08
    cbar=fig2.colorbar(cp,cax=cax, label=r'$B_2^0$',extend='both',pad=1,orientation="vertical")
    cbar.set_label(label=r'',size=30,weight='bold',labelpad=1,loc='center')
    cbar.ax.tick_params(labelsize=27,labelbottom=False,labelleft=True,labelright=False,labeltop=False,right=False,left=True,bottom=False,top=False)
    cbar.ax.yaxis.set_minor_locator(ticker.SymmetricalLogLocator(linthresh=.001,base=10,subs=[2, 3, 4, 5, 6, 7, 8, 9]))
    cbar.ax.tick_params(which='major', length=10,width=2.5)
    cbar.ax.tick_params(which='minor', length=6,width=1.2,left=True,right=False)

#     axes2[jz,2].set_ylabel(r"$B_%d^%d$"%(ell,emm),fontsize=47)
    cbar.set_label(label=r'$\bar{B}$',size=40,weight='bold',labelpad=-120,loc='center',va='bottom')

  
    
ii=0
up_param=[.92,.9,.442,.21] 
cax = fig2.add_axes([0.66,up_param[ii],.185,0.015])  ## right, up, width, height, right=0.283, width=.255
cbar=fig2.colorbar(cp_err,cax=cax, label=r'$B_0^0$',extend='both',pad=1,orientation="horizontal")
cbar.set_label(label=r'',size=30,weight='bold',labelpad=1,loc='center')
cbar.ax.tick_params(labelsize=25,rotation=0)
#     cbar.ax.xaxis.set_minor_locator(ticker.SymmetricalLogLocator(linthresh=.0001,base=10,subs=[2, 3, 4, 5, 6, 7, 8, 9]))
cbar.ax.tick_params(which='major', length=10,width=2.5)
cbar.ax.tick_params(which='minor', length=6,width=1.2)
cbar.set_label(label=r'$\Delta _l^m$',size=30,weight='bold',labelpad=8,loc='center')

####################################################################################
    
fnt_sz=24
for jz in range(4):
    labelkeyy=1 if jz==3 else 0
    axes2[jz][0].xaxis.set_tick_params(which='both', top=True, labeltop=False,
                             bottom=True, labelbottom=labelkeyy, labelsize=fnt_sz)
    axes2[jz][0].yaxis.set_tick_params(which='both', top=True, labeltop=False,
                             bottom=True, labelbottom=1, labelsize=fnt_sz)

#     axes2[jz][1].xaxis.set_tick_params(which='both', top=True, labeltop=False,
#                              bottom=True, labelbottom=labelkeyy, labelsize=fnt_sz)
#     axes2[jz][1].yaxis.set_tick_params(which='both', top=True, labeltop=False,
#                              bottom=0, labelbottom=0, labelsize=fnt_sz)

    axes2[jz][1].xaxis.set_tick_params(which='both', top=True, labeltop=False,
                             bottom=True, labelbottom=labelkeyy, labelsize=fnt_sz)
    axes2[jz][1].yaxis.set_tick_params(which='both', top=True, labeltop=False,
                             bottom=0, labelbottom=0, labelsize=fnt_sz)

    axes2[jz][0].set_xticks([.507,.6,.7,.8,.9,.993])
    axes2[jz][0].set_xticklabels(['0.5','0.6','0.7','0.8','0.9','1'])
    axes2[jz][1].set_xticks([.507,.6,.7,.8,.9,1])
    axes2[jz][1].set_xticklabels(['0.5','0.6','0.7','0.8','0.9','1'])
    axes2[jz][0].set_yticks([.507,.6,.7,.8,.9,.993])
    axes2[jz][0].set_yticklabels(['0.5','0.6','0.7','0.8','0.9','1'])
    axes2[jz][1].set_yticks([.507,.6,.7,.8,.9,.993])
    axes2[jz][1].set_yticklabels(['0.5','0.6','0.7','0.8','0.9','1'])
    
    
    
axes2[3][0].set_xlabel("$\mu$",fontsize=45)
# axes2[3][1].set_xlabel("$\mu$",fontsize=45)
axes2[3][1].set_xlabel("$\mu$",fontsize=45)
for jz in range(4):
    axes2[jz][0].set_ylabel("$t$",fontsize=45)

    
fnt_sz=35
jz=0
axes2[jz][0].text(.525, .55, r'$\bar{B}$', fontsize=fnt_sz, color='black')
axes2[jz][1].text(.525, .55, r'$B_{Ana}$', fontsize=fnt_sz, color='black')
axes2[jz][3].text(.525, .55, r'$\Delta$', fontsize=fnt_sz, color='black')
# jz=1
# axes2[jz][0].text(.525, .55, r'$\bar{B}_2^0$', fontsize=fnt_sz, color='black')
# # axes2[jz][1].text(.525, .55, r'$\Delta {B_2^0}/[\bar{B}_2^0]_d$', fontsize=fnt_sz, color='black')
# axes2[jz][1].text(.525, .55, r'$\Delta _2^0$', fontsize=fnt_sz, color='black')
# jz=2
# axes2[jz][0].text(.525, .55, r'$\bar{B}_2^1$', fontsize=fnt_sz, color='black')
# # axes2[jz][1].text(.525, .55, r'$\Delta {B_2^1}/[\bar{B}_2^1]_d$', fontsize=fnt_sz, color='black')
# axes2[jz][1].text(.525, .55, r'$\Delta _2^1$', fontsize=fnt_sz, color='black')
# jz=3
# axes2[jz][0].text(.525, .55, r'$\bar{B}_2^2$', fontsize=fnt_sz, color='black')
# # axes2[jz][1].text(.525, .55, r'$\Delta {B_2^2}/[\bar{B}_2^2]_d$', fontsize=fnt_sz, color='black')
# axes2[jz][1].text(.525, .55, r'$\Delta _2^2$', fontsize=fnt_sz, color='black')
    

x=np.linspace(.5,1,50)
y=1/(2*x)
for jz in [0,1]:
    for lz in range(4):
        axes2[lz][jz].plot(x,y,"--",color="black",linewidth=1.7)
             


In [ ]:
tri_u1[7]/1e5


In [ ]:
# LL=np.pi*1.719/(180*60)
LL=1.719/(60)
2*np.pi/(4096*LL)

In [ ]:
thx=24.68
th0=.6*thx
np.sqrt(np.log(2))/(th0*np.pi)

In [ ]:
1/np.deg2rad(1/0.0179)

In [ ]:
(.62-.58)/.58

In [ ]:
1217/tpi